[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_03_jit_static.ipynb)

# 🟢 Easy: jit with static_argnames

*JAX Fundamentals*
Implement a **masked mean**: average `x` along `axis`, counting only the
positions where `mask` is `True`.

$$\text{out} = \frac{\sum_i x_i \cdot m_i}{\max\left(\sum_i m_i,\; 1\right)}$$

### Rules
- The function must be wrapped in `jax.jit`
- `axis` must be a **static** argument (`static_argnames`) — it determines the
  output shape, so JAX cannot trace it as a value
- A slice where the mask is entirely `False` must return `0.0`, **not** `NaN`
- `mask` is a boolean array broadcastable to `x`

### Signature
```python
@partial(jax.jit, static_argnames=("axis",))
def masked_mean(x, mask, axis):
    ...
```

### Why it matters
This is the single most common `jit` gotcha. Anything that affects a **shape**
— an axis, a `k`, a boolean that picks a branch returning different shapes —
must be static. Anything that is merely a **value** should stay traced, because
every distinct static argument triggers a fresh XLA compilation.

Padding-masked means show up constantly in real transformer code: averaging
token embeddings while ignoring `<pad>` positions.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

from functools import partial

import jax
import jax.numpy as jnp


@partial(jax.jit, static_argnames=("axis",))
def masked_mean(x, mask, axis):
    """Mean of x over `axis`, counting only positions where mask is True.

    Args:
        x:    float array
        mask: boolean array broadcastable to x
        axis: int (static) — axis to reduce

    Returns:
        Array with `axis` reduced. Fully-masked slices must be 0.0, not NaN.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

x = jnp.array([[1.0, 2.0, 3.0, 4.0],
               [5.0, 6.0, 7.0, 8.0]])
mask = jnp.array([[True, True, False, False],
                  [True, False, False, False]])

print("mean over axis=1:", masked_mean(x, mask, axis=1))  # [1.5, 5.0]
print("mean over axis=0:", masked_mean(x, mask, axis=0))
print("is jitted:", hasattr(masked_mean, "lower"))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("jit_static")

# hint("jit_static")      # stuck? nudge without the answer
# solution("jit_static")  # spoiler: the reference implementation